# Sinh dataset bat dinh da cap (Anh-Viet) cho UTKMLHUI

Notebook nay tai tao dung pipeline da dung de sinh `taxonomy.txt`, `name.txt`,
`utility_probability.txt` (1 file gop) va `k.txt` cho UTKMLHUI, chay doc lap
tren Colab.

**Cac buoc:** cai dat -> build fast_align -> tai corpus -> tien xu ly
(tokenize EN + POS-tag dung ngu canh cau + LEMMATIZE bang WordNet, phan doan
CUNG tieng Viet bang CRF) -> chay fast_align -> loc nhieu -> sinh taxonomy 3
tang + utility (TF-IDF) + probability (p_align).

**Lich su quyet dinh ve tang lemma (tang cao nhat):**
- Ban dau dung Porter Stemmer -- nhanh, nhung sinh ra "stem" (VD: "creat",
  "narr", "evid") thay vi "lemma" that -- 49.5% ket qua khong phai tu tieng
  Anh co nghia. Porter Stemmer dung DE lam dung chuc nang cua no (stem khong
  bat buoc la tu that), nhung khong khop voi thuat ngu "lemma" ma project
  dang dung tu dau.
- Da thu vaa (giu Porter, chi doi ten hien thi thanh bien the ngan nhat trong
  nhom) -- cai thien con 23.4%, nhung van la STEM ve ban chat, khong phai
  LEMMA, va doi khi chon nham tu loai dai dien (VD "evident" thay vi
  "evidence").
- **Quyet dinh cuoi cung: doi han sang WordNetLemmatizer + pos_tag dung ngu
  canh cau** (khong tag tu roi rac, vi lam vay pos_tag de doan sai) -- day
  moi thuc su la LEMMA dung nghia (tu dien that), khop dung thuat ngu du an
  da dung. Doi lai: nhom nho hon (mot tu chi gom cac bien the NGU PHAP, VD
  create/creates/created/creating, khong gom xuyen tu loai voi creation/
  creations nhu Porter tung lam).

**Xac suat p(i,T) = p_align only** (khong con p_seg/p_exist -- xem cac quyet
dinh truoc, ly do la khai niem: tai thoi diem build transaction, tu tieng
Viet dung lam leaf da la ket qua segmenter quyet dinh roi, hoi "co ton tai
khong" la hoi cau da co dap an).




**Cap nhat moi: loc ten rieng bang NER that (spaCy 'en_core_web_sm')** thay
cho heuristic viet hoa+vi tri cu -- heuristic cu KHONG bat duoc ten rieng o
DAU CAU (vi moi tu dau cau deu viet hoa, khong phan biet duoc voi ten rieng
that). NER nhin ca cau lam ngu canh, phan biet dung 'Kathryn Schulz : ...'
(ten nguoi, du o dau cau) voi 'The science...' (khong phai ten). Da benchmark
tren mau nho: ~593 cau/giay, uoc tinh ~3.7 phut cho toan bo 133k cau -- khong
dang ke so voi cac buoc khac trong pipeline.

Chay lan luot tung cell theo thu tu (Runtime > Run all cung duoc).

---

## CAP NHAT LON: chuyen sang P(Anh|Viet) + taxonomy DA-CHA (dang lam phan SCRIPT truoc, Java/bao cao lam SAU)

Theo huong da ban voi Thay: xac suat khong con la p(i,T) theo tung transaction
nua, ma la TRONG SO CANH cha-con trong chinh taxonomy:

```
"con_nguoi" (leaf)
    +--(0.43)--> "people" (variant)
    +--(0.23)--> "human"  (variant)
```

**Thay doi cu the:**
- Leaf gio la **1 TU VIET DUY NHAT** (khong con tach theo tung cap (Anh,Viet)
  nhu truoc -- vi 1 tu Viet co the co NHIEU cha cung luc, khong can tach de
  tranh xung dot cay 1-cha nhu thiet ke cu nua)
- `taxonomy.txt` doi dinh dang: **THEM COT TRONG SO** -- `child,parent,weight`
  (truoc day chi co 2 cot `child,parent`)
- Trong so canh leaf->variant = **P(Anh|Viet)**, tinh bang `fast_align -r`
  (chieu NGUOC so voi truoc day)
- Canh variant->lemma van tat dinh (WordNet lemma), trong so co dinh = 1.0
- `utility_probability.txt` van giu 4 phan nhu cu, nhung cot probability gio
  la **placeholder co dinh = 1.0** cho moi item -- vi bat dinh da chuyen
  sang nam o trong so canh taxonomy, khong con o tang transaction nua

**QUAN TRONG -- viec con treo, CHUA lam trong lan cap nhat nay:**
- `Taxonomy.java` hien tai (`Map<Integer,Integer> parentOf`) **KHONG DOC
  DUOC** dinh dang 3 cot moi nay -- can sua lai thanh cau truc da-cha co
  trong so truoc khi dung duoc voi thuat toan that
- `UTKMLHUIAlgo.java` (cong don EU) can sua tu cong don gian (`+= u`) thanh
  cong co trong so (`+= u * weight`)
- Bao cao: Dinh nghia 3.1 (p(i,T)) va 3.3 (cong don EU) can viet lai, ra soat
  lai Bo de A

Day la quyet dinh CO CHU DICH: lam truoc phan sinh du lieu (script) de co
san dataset dung dinh dang moi, phan Java + bao cao se lam rieng sau.

---

## CAP NHAT: trong so canh variant -> lemma (thay cho 1.0 co dinh truoc day)

Theo dung mach thao luan voi Thay: canh variant->lemma tu no la 2 buoc con
(gan tu loai pos_tag = x, tra dictionary lemmatize = y). Da xac nhan **y
luon = 1.0** (morphy la phep tra bang tat dinh, du co nhieu ung vien cung
chi chon 1 theo quy tac cung "chuoi ngan nhat", khong dung xac suat).

Con **x** -- da thu VA BAC BO cach lay do tin cay `pos_tag` cho 1 CAU cu
the (`return_conf=True`), vi 2 ly do:
1. Epistemic, khong phai aleatory -- vi du "left" o 4 cau khac nhau deu cho
   x~0.999 (rat tu tin) nhung moi cau tu tin vao 1 ket qua khac nhau.
2. Ban chat so hoc: diem tho cua Perceptron qua `softmax` bi "qua tu tin"
   (exponential amplification) -- kiem chung: "ran" co diem tho VBD=31.4 vs
   VBZ=15.0 (chenh lech ~16 diem), nhung sau softmax thanh 0.9999999 vs
   ~0 (chenh nhau ~13 TRIEU lan). Diem tho cung khong dung duoc lam xac
   suat (co gia tri AM, tong gan bang 0).
3. Ly do ky thuat: `taxonomy.txt` la file TINH dung chung moi transaction,
   khong co cho luu "nhieu gia tri cho 1 canh tuy theo cau".

**Da CHOT**: x = tan suat cua tu loai THANG majority-vote, dem qua TOAN
CORPUS (co Laplace smoothing cho tu hiem):
```
x(variant) = (so_lan_thang_majority + 1) / (tong_so_lan_xuat_hien + 2)
```
Vi du da kiem chung: "ran"=0.86, "human"=0.976, "people"=0.999,
"left"=0.376 (rat mo ho, dung nhu du doan).

**Bug da sua truoc khi tinh x**: them `hasn wasn doesn isn aren wouldn
couldn didn shouldn weren` vao STOPWORDS (manh vun tu "wasn't"->"wasn"+"'t"
lam nhieu so dem neu khong loc truoc).

---

## CAP NHAT: chan tieng Viet lot vao phia tieng Anh (loi phat hien tren MTet 150k)

Phat hien: mot so dong du lieu MTet co the bi loi (cot "tieng Anh" chua
nham tieng Viet), khien tu tieng Viet di qua toan bo pipeline xu ly tieng
Anh, sinh ra lemma/variant la tieng Viet (VD "dieu*", "me*"). Nguyen nhan
goc: `is_content_token()` dung chung 1 dai Unicode cho ca 2 ngon ngu, khong
phan biet duoc.

Da them ham `looks_like_vietnamese()` (kiem tra ky tu co dau tieng Viet)
vao dieu kien `usable` o phia tieng Anh. **Gioi han da biet**: chi bat duoc
tu co dau thanh -- cac am tiet tieng Viet KHONG dau (vd "gia", "bay", "hai")
van co the lot qua buoc nay, nhung van phai vuot tiep MIN_EN_FREQ va
MIN_PROB+MIN_LINK_COUNT moi thuc su xuat hien trong taxonomy.txt cuoi cung.


---

## CAP NHAT MOI (2 thay doi lon)

**1) Chon corpus linh hoat (Cell 3)** -- gio chi can sua 2 dong o Cell 3:
```python
CORPUS = "iwslt15"   # hoac "mtet"
N_SENTENCES = None   # None = lay het, hoac vd 150000
```
roi chay lan luot Cell 3 -> 3b -> 4, khong can comment/uncomment thu cong
nhu truoc nua. Ca 2 nguon deu ra cung dinh dang `train.en`/`train.vi`.

**2) variant -> lemma gio la DA-LEMMA (giong tinh than da-cha o tang duoi)**
-- Thay chi doi voi Thay: neu 1 tu (vd "left") THAT SU duoc dung voi ca 2
vai tro (dong tu VA danh tu) qua nhieu cau khac nhau, thi CA HAI deu la
su that (da duoc pos_tag xac dinh dung theo ngu canh tung cau, khong phai
doan mo) -- khong duoc ep chon 1 ben roi bo ben con lai.

Sua lai: mot variant co the co NHIEU dong canh len lemma trong
`taxonomy.txt`, trong so = tan suat LEMMA DICH (khong phai theo nhan tho --
da phat hien bug: neu gop theo nhan tho, "ran" (VBD=18,VBP=2, nhung CA HAI
deu ra cung 1 lemma "run") se bi tinh SAI thanh trong so 0.9 thay vi dung
phai la 1.0). Vi du da kiem chung tren du lieu that:
```
"ran"  -> chi 1 lemma "run*", trong so = 1.0 (dung, vi VBD/VBP cung ra 1 lemma)
"left" -> 2 lemma that: "left*" (0.494) va "leave*" (0.506), tong = 1.0
```


In [1]:
# ==== 1. Cai dat he thong + thu vien Python ====
!apt-get update -qq
!apt-get install -y -qq cmake libgoogle-perftools-dev libsparsehash-dev
!pip install -q underthesea nltk spacy
!python -m spacy download en_core_web_sm -q

import nltk
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

print("Xong buoc 1.")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libunwind-dev:amd64.
(Reading database ... 122797 files and directories currently installed.)
Preparing to unpack .../libunwind-dev_1.6.2-3build1.1_amd64.deb ...
Unpacking libunwind-dev:amd64 (1.6.2-3build1.1) ...
Selecting previously unselected package libgoogle-perftools-dev:amd64.
Preparing to unpack .../libgoogle-perftools-dev_2.15-3build1_amd64.deb ...
Unpacking libgoogle-perftools-dev:amd64 (2.15-3build1) ...
Selecting previously unselected package libsparsehash-dev.
Preparing to unpack .../libsparsehash-dev_2.0.3-2_all.deb ...
Unpacking libsparsehash-dev (2.0.3-2) ...
Setting up libunwind-dev:amd64 (1.6.2-3build1.1) ...
Setting up libgoogle-perftools-dev:amd64 (2.15-3build1) ...
Setting up libsparsehash-dev (2.0.3-2) ...
Processing triggers for man-db (2

In [2]:
# ==== 2. Clone va build fast_align (Dyer, Chahuneau & Smith, 2013) ====
%cd /content
!git clone --depth 1 https://github.com/clab/fast_align.git
%cd /content/fast_align
!mkdir -p build && cd build && cmake .. -DCMAKE_BUILD_TYPE=Release && make -j4
print("Xong buoc 2. Binary tai /content/fast_align/build/fast_align")


/content
Cloning into 'fast_align'...
remote: Enumerating objects: 21, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 21 (delta 0), reused 7 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (21/21), 22.41 KiB | 2.49 MiB/s, done.
/content/fast_align
CMake Warning (dev) at CMakeLists.txt:1 (project):
  cmake_minimum_required() should be called prior to this top-level project()
  call.  Please see the cmake-commands(7) manual for usage documentation of
  both commands.
This warning is for project developers.  Use -Wno-dev to suppress it.

-- The C compiler identification is GNU 13.3.0
-- The CXX compiler identification is GNU 13.3.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check 

In [3]:
# ==== 3. Cau hinh: chon corpus va so cau muon dung ====
CORPUS = "mtet"     # "iwslt15" (mac dinh, 133,317 cau) hoac "mtet" (toi da 8,327,706 cau)
N_SENTENCES = 150000     # None = lay HET corpus da chon; hoac 1 so cu the, vd 150000, de gioi han
                       # (voi "mtet", nen dat 1 so cu the -- lay het 8.3 trieu cau se rat lau)


In [4]:
# ==== 3b. Tai corpus theo lua chon o cell tren ====
import os
os.makedirs('/content/uncertain_dataset', exist_ok=True)
%cd /content/uncertain_dataset

if CORPUS == "iwslt15":
    print("Dang tai IWSLT15 English-Vietnamese...")
    !curl -sL -o train-en-vi.tgz "https://github.com/stefan-it/nmt-en-vi/raw/master/data/train-en-vi.tgz"
    !tar -xzf train-en-vi.tgz
    print("Da tai xong IWSLT15.")

elif CORPUS == "mtet":
    print("Dang tai MTet (phongmt184172/mtet)...")
    # Schema thuc te da xac nhan: ex['translation']['source'] = tieng Viet,
    #                              ex['translation']['target'] = tieng Anh
    !pip install -q datasets
    from datasets import load_dataset
    ds = load_dataset('phongmt184172/mtet', split='train')
    print(ds)
    with open('train.en', 'w', encoding='utf-8') as fe, \
         open('train.vi', 'w', encoding='utf-8') as fv:
        for i, ex in enumerate(ds):
            if N_SENTENCES and i >= N_SENTENCES:
                break
            fe.write(ex['translation']['target'].strip() + '\n')
            fv.write(ex['translation']['source'].strip() + '\n')
    print("Da xuat train.en / train.vi tu MTet.")

else:
    raise ValueError(f"CORPUS khong hop le: '{CORPUS}' -- chi nhan 'iwslt15' hoac 'mtet'")

!wc -l train.en train.vi


/content/uncertain_dataset
Dang tai MTet (phongmt184172/mtet)...


README.md:   0%|          | 0.00/206 [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

data/train-00000-of-00014.parquet: reconstructing file:   0%|          |  0.00B /  320MB            

data/train-00000-of-00014.parquet: downloading bytes:           |  0.00B            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/8327706 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3106 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2536 [00:00<?, ? examples/s]

Dataset({
    features: ['prompt', 'response', 'source', 'translation'],
    num_rows: 8327706
})
Da xuat train.en / train.vi tu MTet.
  150000 train.en
  150000 train.vi
  300000 total


In [5]:
# ==== 4. Cat lay dung so cau da chon o Cell 3 (ap dung cho ca 2 nguon corpus) ====
%cd /content/uncertain_dataset
import subprocess
total = int(subprocess.run(['wc', '-l', 'train.en'], capture_output=True, text=True).stdout.split()[0])
n = N_SENTENCES or total
n = min(n, total)
print(f"Tong so cau co san trong file da tai: {total}. Se dung: {n}")

!head -{n} train.en > sub.en
!head -{n} train.vi > sub.vi
!wc -l sub.en sub.vi


/content/uncertain_dataset
Tong so cau co san trong file da tai: 150000. Se dung: 150000
  150000 sub.en
  150000 sub.vi
  300000 total


In [6]:
# ==== 5. Tien xu ly: NER loc ten rieng + tokenize EN (chan ca tieng Viet lot vao) + lemmatize + phan doan Viet ====
%cd /content/uncertain_dataset

import re, json, html
from nltk import pos_tag
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet as wn
import spacy
from spacy.tokens import Doc
from underthesea.pipeline.word_tokenize.regex_tokenize import tokenize as vi_raw_tokenize
from underthesea.models.fast_crf_sequence_tagger import FastCRFSequenceTagger
from os.path import dirname, join
import underthesea.pipeline.word_tokenize as wtmod

wnl = WordNetLemmatizer()

STOPWORDS = set("""
a an the of in on at to for with by from as is are was were be been being
this that these those it its it's he she they we you i him her them us
his hers their our your my and or but if not no so than then there here
do does did doing have has had having will would shall should can could
may might must about into over under again further once out up down off
's 'd 're 'll 've 'm 't
hasn wasn doesn isn aren wouldn couldn didn shouldn weren
against although because during else anything
""".split())

def get_wordnet_pos(tag):
    if tag.startswith('J'): return wn.ADJ
    if tag.startswith('V'): return wn.VERB
    if tag.startswith('R'): return wn.ADV
    return wn.NOUN

# ==== NER that (spaCy) de loc ten rieng, thay the/bo sung cho heuristic viet hoa+vi tri cu ====
_ner_model = None
def get_ner_model():
    global _ner_model
    if _ner_model is None:
        _ner_model = spacy.load('en_core_web_sm', disable=['parser', 'lemmatizer', 'tagger'])
    return _ner_model

def batch_person_masks(list_of_token_lists):
    """Chay NER theo LO (batch) tren danh sach cau da tokenize san (moi cau la 1 list token).
    Tra ve list cac mask (list bool), True = token nam trong 1 thuc the PERSON."""
    nlp = get_ner_model()
    docs = [Doc(nlp.vocab, words=toks) for toks in list_of_token_lists]
    for name, proc in nlp.pipeline:
        docs = list(proc.pipe(docs))
    masks = []
    for doc in docs:
        mask = [False] * len(doc)
        for ent in doc.ents:
            if ent.label_ == 'PERSON':
                for i in range(ent.start, ent.end):
                    mask[i] = True
        masks.append(mask)
    return masks

_model = None
def get_vi_model():
    global _model
    if _model is None:
        _model = FastCRFSequenceTagger()
        wd = dirname(wtmod.__file__)
        _model.load(join(wd, "models", "ws_crf_vlsp2013_20230727"))
    return _model

def hard_segment(sentence):
    model = get_vi_model()
    raw_tokens = vi_raw_tokenize(sentence, use_token_normalize=True, fixed_words=[])
    if not raw_tokens:
        return []
    features = model.featurizer.process([[[t] for t in raw_tokens]])[0]
    tags = model.estimator.tag(features)
    raw_lower = [t.lower() for t in raw_tokens]
    words = []
    cur = []
    for tok, tag in zip(raw_lower, tags):
        if tag == 'I-W' and cur:
            cur.append(tok)
        else:
            if cur:
                words.append('_'.join(cur))
            cur = [tok]
    if cur:
        words.append('_'.join(cur))
    return words

def tok_en_with_case(line):
    return [t for t in line.strip().split() if t.strip()]

def is_content_token(t):
    return bool(re.search(r'[a-zA-Z\u00C0-\u1EF9]', t))

def looks_proper_noun_heuristic(tok, position_in_sentence):
    # Heuristic cu (viet hoa + khong o dau cau) -- van giu lam lop loc PHU,
    # vi no re va bat duoc mot so truong hop NER co the bo sot.
    if position_in_sentence == 0:
        return False
    return tok[0].isupper() and tok[1:].islower()

def looks_apostrophe_fragment(tok):
    # Corpus nay tokenize moi tu co dau nhay don thanh 2 manh (VD "didn" + "'t",
    # "y" + "'all", "o" + "'clock"...) -- BAT KY token nao bat dau bang dau nhay
    # don deu la manh vun, khong phai tu that. Quy tac tong quat, khong lam theo
    # tung truong hop cu the (danh sach thu cong se khong bao gio du).
    return tok.startswith("'") or tok.startswith("\u2019")

def looks_like_vietnamese(tok):
    # Phia tieng Anh KHONG duoc chua ky tu co dau tieng Viet -- neu co, chac
    # chan la loi du lieu (VD MTet co dong bi lech cot Anh/Viet), khong phai
    # tu tieng Anh that. Luu y: KHONG ap dung quy tac nguoc lai cho phia Viet
    # (leaf Viet duoc phep chua tu muon tieng Anh nhu "email","video" -- do la
    # code-switching hop le, khong phai loi).
    return bool(re.search(r'[\u00C0-\u1EF9]', tok))

with open('sub.en', encoding='utf-8') as f:
    en_lines = [l.rstrip('\n') for l in f]
with open('sub.vi', encoding='utf-8') as f:
    vi_lines = [l.rstrip('\n') for l in f]
assert len(en_lines) == len(vi_lines)

# ==== BUOC 0: unescape HTML + tokenize truoc, de chay NER theo LO cho nhanh ====
en_lines = [html.unescape(l) for l in en_lines]
vi_lines = [html.unescape(l) for l in vi_lines]
raw_en_all = [tok_en_with_case(l) for l in en_lines]

print("Dang chay NER (spaCy) theo lo tren toan bo cau tieng Anh...")
person_masks_all = batch_person_masks(raw_en_all)
print("Xong NER.")

en_tok_sents = []
en_lemma_sents = []
vi_tok_sents = []
en_mask = []

for raw_en, vi, person_mask in zip(raw_en_all, vi_lines, person_masks_all):
    words_vi = hard_segment(vi)
    if not raw_en or not words_vi:
        continue
    lowered_en = [t.lower() for t in raw_en]

    mask = []
    for pos, (raw_t, low_t, is_person) in enumerate(zip(raw_en, lowered_en, person_mask)):
        usable = (is_content_token(low_t) and low_t not in STOPWORDS
                  and not looks_proper_noun_heuristic(raw_t, pos)
                  and not looks_apostrophe_fragment(low_t)
                  and not looks_like_vietnamese(low_t)   # <- MOI: chan tieng Viet lot vao phia Anh
                  and not is_person)   # <- NER: chan ten rieng CA O DAU CAU
        mask.append(usable)

    tagged = pos_tag(lowered_en)
    lemmas = [wnl.lemmatize(w, get_wordnet_pos(tag)) for w, tag in tagged]

    en_tok_sents.append(lowered_en)
    en_lemma_sents.append(lemmas)
    vi_tok_sents.append(words_vi)
    en_mask.append(mask)

print(f"Sentences kept: {len(en_tok_sents)}")

with open('align_input.txt', 'w', encoding='utf-8') as f:
    for et, vt in zip(en_tok_sents, vi_tok_sents):
        f.write(' '.join(et) + ' ||| ' + ' '.join(vt) + '\n')

with open('tok_cache.json', 'w', encoding='utf-8') as f:
    json.dump({'en': en_tok_sents, 'en_lemma': en_lemma_sents, 'vi': vi_tok_sents, 'en_mask': en_mask}, f, ensure_ascii=False)

print("OK")


/content/uncertain_dataset
Dang chay NER (spaCy) theo lo tren toan bo cau tieng Anh...
Xong NER.
Sentences kept: 149991
OK


In [7]:
# ==== 6. Chay fast_align CHIEU NGUOC (-r) de lay P(Anh | Viet) -- dung lam trong so canh taxonomy da-cha ====
%cd /content/uncertain_dataset
import subprocess, time
t0 = time.time()
r = subprocess.run(
    ['/content/fast_align/build/fast_align', '-i', 'align_input.txt', '-r', '-d', '-o', '-v', '-p', 'ttable_reverse.txt'],
    stdout=open('alignments.txt', 'w'), stderr=open('align.log', 'w')
)
print(f"fast_align (chieu nguoc) chay xong trong {time.time()-t0:.1f}s (returncode={r.returncode})")
!tail -8 align.log


/content/uncertain_dataset
fast_align (chieu nguoc) chay xong trong 379.0s (returncode=0)
  log_e likelihood: -1.94416e+07
  log_2 likelihood: -2.80483e+07
     cross entropy: 7.28178
        perplexity: 155.609
      posterior p0: 0
 posterior al-feat: 0
       size counts: 9014
conditional probabilities: ttable_reverse.txt


In [8]:
# ==== 7. Sinh taxonomy DA-CHA CA 2 TANG (leaf->variant: P(Anh|Viet); variant->lemma: co the NHIEU lemma, trong so = tan suat lemma dich, gop corpus) ====
%cd /content/uncertain_dataset

import json, math, re
from collections import defaultdict, Counter

MIN_PROB = 0.05
MIN_EN_FREQ = 3
VI_STOPWORDS = set("""
của là các những về và một này đã sẽ cho được để với mà thì vì nên
cũng rằng khi ra lên vào còn nếu như đến từ trong ngoài trên dưới
không có ở đó đây ấy nào ai gì bao nhiêu vẫn đang chỉ chưa nữa lại
nhưng hay hoặc nhất rất khá quá lắm thật sự việc cái con người ta
ta_-- tôi bạn anh chị em nó họ mình
""".split())

with open('tok_cache.json', encoding='utf-8') as f:
    cache = json.load(f)
en_sents = cache['en']
en_lemma_sents = cache['en_lemma']
vi_sents = cache['vi']
en_mask = cache['en_mask']

with open('alignments.txt', encoding='utf-8') as f:
    align_lines = [l.strip() for l in f]
assert len(align_lines) == len(en_sents) == len(vi_sents) == len(en_mask) == len(en_lemma_sents)

# BANG XAC SUAT CHIEU NGUOC: P(Anh | Viet) -- day la TRONG SO CANH cha-con moi
ttable_reverse = {}
with open('ttable_reverse.txt', encoding='utf-8') as f:
    for line in f:
        parts = line.rstrip('\n').split('\t')
        if len(parts) != 3:
            continue
        src, tgt, logp = parts   # voi -r: src=Viet, tgt=Anh
        if src == '<eps>':
            continue
        try:
            p = math.exp(float(logp))
        except ValueError:
            continue
        ttable_reverse[(src, tgt)] = p   # key = (vi_w, en_w) -> P(en_w | vi_w)

MIN_LINK_COUNT = 3  # canh (en,vi) phai xuat hien THAT it nhat 3 lan qua alignment moi duoc giu

def is_content_token(t):
    return bool(re.search(r'[a-zA-Z\u00C0-\u1EF9]', t))

en_freq = Counter()
for et, mask in zip(en_sents, en_mask):
    for w, usable in zip(et, mask):
        if usable and is_content_token(w):
            en_freq[w] += 1
allowed_en = {w for w, c in en_freq.items() if c >= MIN_EN_FREQ}
print(f"Distinct usable EN words: {len(en_freq)}; kept after freq>={MIN_EN_FREQ}: {len(allowed_en)}")

lemma_votes = defaultdict(Counter)
for et, lemmas, mask in zip(en_sents, en_lemma_sents, en_mask):
    for w, lm, usable in zip(et, lemmas, mask):
        if usable and is_content_token(w):
            lemma_votes[w][lm] += 1

# ==== TRONG SO CANH variant -> lemma: KHONG ep ve 1 lemma duy nhat nua.
#      Theo gop y cua Thay: viec xac dinh tu loai cho 1 occurrence CU THE
#      (qua pos_tag dung ngu canh cau) da la ket qua DUNG, dang tin cay --
#      neu "left" that su co luc la dong tu (-> lemma "leave"), co luc la
#      danh tu (-> lemma "left"), thi CA HAI deu la su that, khong duoc ep
#      chon 1 ben roi vut bo ben con lai (mat thong tin that). Giu CA HAI
#      lemma, moi cai mang 1 trong so = tan suat xuat hien cua dung LEMMA
#      DICH do (khong phai theo nhan tho -- vi nhieu nhan tho khac nhau,
#      VD "VBD","VBP", co the CUNG ra 1 lemma nhu nhau, vd "ran" -> "run" --
#      da kiem chung neu gop nham theo nhan se tinh sai trong so).
#      Dung Laplace smoothing tong quat (add-1, chia deu cho so lemma dich
#      khac nhau) cho tu hiem: neu 1 variant chi co DUNG 1 lemma dich duy
#      nhat (vd "ran"->chi "run"), cong thuc nay cho trong so ~1.0 dung
#      nhu ky vong, khong con bi tinh sai thanh 0.9 nhu truoc day. ====
ALPHA = 1.0
variant_to_lemma_weights = {}   # variant -> {lemma: weight}
for w, counter in lemma_votes.items():
    total = sum(counter.values())
    k_distinct = len(counter)
    variant_to_lemma_weights[w] = {
        lm: round((count + ALPHA) / (total + ALPHA * k_distinct), 6)
        for lm, count in counter.items()
    }

# ==== BUOC DEM: so lan cap (en,vi) THAT SU xuat hien qua alignment (khong tinh theo xac suat) ====
link_count = Counter()
for aline, et, vt, mask in zip(align_lines, en_sents, vi_sents, en_mask):
    for link in aline.split():
        if '-' not in link:
            continue
        a, b = link.split('-')
        a, b = int(a), int(b)
        if a >= len(et) or b >= len(vt):
            continue
        if not mask[a]:
            continue
        en_w, vi_w = et[a], vt[b]
        if not is_content_token(en_w) or not is_content_token(vi_w):
            continue
        if vi_w in VI_STOPWORDS:
            continue
        link_count[(en_w, vi_w)] += 1

# ==== LEAF = 1 TU VIET DUY NHAT (khong tach theo cap en,vi nua) ====
leaf_ids = {}          # vi_w -> id
variant_ids = {}       # en_w -> id
lemma_ids = {}         # lemma -> id
leaf_parent_edges = defaultdict(dict)   # leaf_id -> {variant_id: weight}  (DA-CHA)
transactions = []
n_links_total = n_links_kept = 0

for i, aline in enumerate(align_lines):
    et = en_sents[i]
    vt = vi_sents[i]
    mask = en_mask[i]
    seen_leaf_ids = set()   # transaction chi can biet leaf nao CO MAT, khong can gia tri xac suat rieng nua
    for link in aline.split():
        if '-' not in link:
            continue
        a, b = link.split('-')
        a, b = int(a), int(b)
        if a >= len(et) or b >= len(vt):
            continue
        n_links_total += 1
        en_w, vi_w = et[a], vt[b]
        if not mask[a]:
            continue
        if not is_content_token(en_w) or not is_content_token(vi_w):
            continue
        if vi_w in VI_STOPWORDS:
            continue
        if en_w not in allowed_en:
            continue
        p_rev = ttable_reverse.get((vi_w, en_w))   # P(en_w | vi_w)
        if p_rev is None or p_rev < MIN_PROB:
            continue
        if link_count[(en_w, vi_w)] < MIN_LINK_COUNT:   # MOI: chan nhieu do trung hop ngau nhien (xuat hien qua it lan)
            continue
        n_links_kept += 1

        if vi_w not in leaf_ids:
            leaf_ids[vi_w] = len(leaf_ids) + 1
        if en_w not in variant_ids:
            variant_ids[en_w] = None
        for lemma in variant_to_lemma_weights.get(en_w, {}):   # co the NHIEU lemma cho 1 variant
            if lemma not in lemma_ids:
                lemma_ids[lemma] = None

        lid = leaf_ids[vi_w]
        seen_leaf_ids.add(lid)
        # canh (leaf -> variant) co trong so P(en_w|vi_w) -- neu da co roi (tu cau khac) thi GIU GIA TRI (khong doi, vi la hang so toan corpus)
        leaf_parent_edges[lid][en_w] = round(p_rev, 6)

    if seen_leaf_ids:
        transactions.append((i, seen_leaf_ids))

print(f"Alignment links: total={n_links_total} kept={n_links_kept} ({100*n_links_kept/max(1,n_links_total):.1f}%)")
print(f"Transactions kept: {len(transactions)}")
print(f"Leaf items (tu Viet duy nhat): {len(leaf_ids)}  Variant items: {len(variant_ids)}  Lemma groups: {len(lemma_ids)}")

next_id = len(leaf_ids) + 1
for w in variant_ids:
    variant_ids[w] = next_id; next_id += 1
for s in lemma_ids:
    lemma_ids[s] = next_id; next_id += 1

def lemma_display_name(lemma):
    return f"{lemma}*"

id_to_vi = {v: k for k, v in leaf_ids.items()}

doc_tokens = []
for i, _ in transactions:
    toks = [t for t in vi_sents[i] if is_content_token(t)]
    doc_tokens.append(toks)

N = len(doc_tokens)
df = Counter()
for toks in doc_tokens:
    for w in set(toks):
        df[w] += 1

def tfidf(vi_w, toks):
    tf = toks.count(vi_w) / max(1, len(toks))
    idf = math.log((N + 1) / (1 + df[vi_w])) + 1.0
    return tf * idf

# ==== taxonomy.txt: DA-CHA, dinh dang MOI "child,parent,weight" ====
with open('taxonomy.txt', 'w', encoding='utf-8') as ftax, open('name.txt', 'w', encoding='utf-8') as fname:
    for lid, vi_w in id_to_vi.items():
        fname.write(f"@ITEM={lid}={vi_w}\n")
        for en_w, weight in leaf_parent_edges[lid].items():
            vid = variant_ids[en_w]
            ftax.write(f"{lid},{vid},{weight}\n")   # <-- CO THEM COT TRONG SO

    written_lemma_name = set()
    for en_w, vid in variant_ids.items():
        fname.write(f"@ITEM={vid}={en_w}\n")
        for lemma, weight in variant_to_lemma_weights.get(en_w, {}).items():
            sid = lemma_ids[lemma]
            ftax.write(f"{vid},{sid},{weight}\n")   # <-- CO THE NHIEU DONG cho cung 1 variant (da-lemma)
            if sid not in written_lemma_name:
                fname.write(f"@ITEM={sid}={lemma_display_name(lemma)} [lemma]\n")
                written_lemma_name.add(sid)

# ==== utility_probability.txt: GIU 4 PHAN nhu cu, nhung probability = 1.0 co dinh
#      (vi bat dinh gio nam o TRONG SO CANH taxonomy, khong con o tang transaction nua
#       -- day la DIEM CAN BAN/THAY QUYET DINH THEM khi sua Java, xem ghi chu) ====
with open('utility_probability.txt', 'w', encoding='utf-8') as fout:
    for tidx, (i, leaf_id_set) in enumerate(transactions):
        toks = doc_tokens[tidx]
        ids = sorted(leaf_id_set)
        utils = [round(tfidf(id_to_vi[lid], toks) * 100, 4) for lid in ids]
        probs = [1.0 for _ in ids]   # placeholder -- xem ghi chu tren
        twu = round(sum(utils), 4)
        fout.write(
            ' '.join(map(str, ids)) + ' : ' + str(twu) + ' : '
            + ' '.join(map(str, utils)) + ' : ' + ' '.join(map(str, probs)) + '\n'
        )

with open('k.txt', 'w', encoding='utf-8') as fk:
    fk.write('10\n')

print("Done.")


/content/uncertain_dataset
Distinct usable EN words: 96290; kept after freq>=3: 28427
Alignment links: total=3745110 kept=457307 (12.2%)
Transactions kept: 107887
Leaf items (tu Viet duy nhat): 11722  Variant items: 10320  Lemma groups: 9366
Done.


In [9]:
# ==== 8. Kiem tra nhanh chat luong ====
%cd /content/uncertain_dataset
print("HTML entity con sot (phai = 0):")
!grep -c "&quot;\|&apos;" name.txt || echo 0

print()
print("Bug tokenize apostrophe con sot (phai = 0):")
!grep -c "^@ITEM=.*='d$\|^@ITEM=.*='t$\|^@ITEM=.*='re$" name.txt || echo 0

print()
print("Vai dong dau taxonomy.txt / name.txt:")
!head -5 taxonomy.txt
!head -5 name.txt

print()
print("Vai lemma vi du (kiem tra co phai tu that khong):")
!grep "\[lemma\]" name.txt | head -10

print()
print("Vai dong dau utility_probability.txt (items : TWU : utilities : probabilities):")
!head -2 utility_probability.txt


/content/uncertain_dataset
HTML entity con sot (phai = 0):
0
0

Bug tokenize apostrophe con sot (phai = 0):
0
0

Vai dong dau taxonomy.txt / name.txt:
1,11723,0.275577
1,14075,0.056285
2,11724,0.151545
2,17263,0.340058
3,11725,0.56945
@ITEM=1=nhớ
@ITEM=2=xung_đột
@ITEM=3=thời_tiết
@ITEM=4=nhanh
@ITEM=5=đột_nhiên

Vai lemma vi du (kiem tra co phai tu that khong):
@ITEM=22043=remember* [lemma]
@ITEM=22044=conflict* [lemma]
@ITEM=22045=weather* [lemma]
@ITEM=22046=fast* [lemma]
@ITEM=22047=suddenly* [lemma]
@ITEM=22048=leg* [lemma]
@ITEM=22049=legs* [lemma]
@ITEM=22050=tire* [lemma]
@ITEM=22051=metallic* [lemma]
@ITEM=22052=ears,* [lemma]

Vai dong dau utility_probability.txt (items : TWU : utilities : probabilities):
1 2 : 164.5888 : 70.3891 94.1997 : 1.0 1.0
3 : 157.0474 : 157.0474 : 1.0


In [10]:
# ==== 9. Nen va tai ve may ====
%cd /content/uncertain_dataset
!zip -q uncertain_dataset_output.zip taxonomy.txt name.txt utility_probability.txt k.txt
from google.colab import files
files.download('uncertain_dataset_output.zip')


/content/uncertain_dataset


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
# ==== 10. Tra cuu: nhap tu tieng Viet (hoac ID) -> hien TAT CA CAC CHA (da-cha) kem trong so ====
def load_query_data_v2():
    id_to_name = {}
    name_to_ids = {}
    with open('name.txt', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line.startswith('@ITEM='):
                continue
            body = line[len('@ITEM='):]
            i = body.index('=')
            nid, nm = int(body[:i]), body[i+1:]
            id_to_name[nid] = nm
            base_name = nm.split(' (nghia cua')[0].rstrip(' [lemma]').strip()
            name_to_ids.setdefault(base_name, []).append(nid)

    # taxonomy gio la DA-CHA: 1 child co the co NHIEU dong (nhieu cha khac nhau)
    parents_of = {}   # child_id -> list[(parent_id, weight)]
    with open('taxonomy.txt', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(',')
            c, p, w = int(parts[0]), int(parts[1]), float(parts[2])
            parents_of.setdefault(c, []).append((p, w))

    return id_to_name, name_to_ids, parents_of

def resolve_ids_v2(query, id_to_name, name_to_ids):
    try:
        qid = int(query)
        if qid in id_to_name:
            return [qid]
    except ValueError:
        pass
    return name_to_ids.get(query, [])

def show_all_parents(query):
    id_to_name, name_to_ids, parents_of = load_query_data_v2()
    ids = resolve_ids_v2(query, id_to_name, name_to_ids)
    if not ids:
        print(f"Khong tim thay item nao khop voi '{query}'")
        return

    for cid in ids:
        cname = id_to_name[cid]
        print(f"\n=== Item id={cid}: '{cname}' ===")
        direct_parents = parents_of.get(cid, [])
        if not direct_parents:
            print("  (khong co cha nao -- day la node goc)")
            continue
        print(f"  Co {len(direct_parents)} cha truc tiep:")
        for pid, weight in sorted(direct_parents, key=lambda x: -x[1]):
            pname = id_to_name[pid].replace(' [lemma]', '')
            print(f"    -> id={pid} '{pname}'   trong so = {weight}")
            # leo tiep len ong (variant -> lemma), thuong chi co 1 cha duy nhat o tang nay
            grandparents = parents_of.get(pid, [])
            for gpid, gweight in grandparents:
                gname = id_to_name[gpid].replace(' [lemma]', '')
                print(f"         -> id={gpid} '{gname}'   trong so = {gweight}")

# ==== Nhap tu tieng Viet (hoac ID so) o day ====
show_all_parents("lá")   # doi thanh tu/ID ban muon tra cuu



=== Item id=7066: 'lá' ===
  Co 1 cha truc tiep:
    -> id=18097 'leaves'   trong so = 0.159053
         -> id=27919 'leaf*'   trong so = 0.546875
         -> id=22916 'leave*'   trong so = 0.453125


In [17]:
# ==== Tra cứu: nhập lemma (hoặc ID) -> hiện các biến thể con, và các nghĩa Việt cháu (kèm trọng số) ====
def load_query_data_v3():
    id_to_name = {}
    name_to_ids = {}
    with open('name.txt', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line.startswith('@ITEM='):
                continue
            body = line[len('@ITEM='):]
            i = body.index('=')
            nid, nm = int(body[:i]), body[i+1:]
            id_to_name[nid] = nm
            base_name = nm.split(' (nghia cua')[0].rstrip(' [lemma]').strip()
            name_to_ids.setdefault(base_name, []).append(nid)

    children_of = {}   # parent_id -> list[(child_id, weight)]
    with open('taxonomy.txt', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(',')
            c, p, w = int(parts[0]), int(parts[1]), float(parts[2])
            children_of.setdefault(p, []).append((c, w))

    return id_to_name, name_to_ids, children_of

def resolve_ids_v3(query, id_to_name, name_to_ids):
    try:
        qid = int(query)
        if qid in id_to_name:
            return [qid]
    except ValueError:
        pass
    return name_to_ids.get(query, [])

def show_children_of_lemma(query):
    id_to_name, name_to_ids, children_of = load_query_data_v3()
    ids = resolve_ids_v3(query, id_to_name, name_to_ids)
    if not ids:
        print(f"Không tìm thấy item nào khớp với '{query}'")
        return

    for pid in ids:
        pname = id_to_name[pid]
        print(f"\n=== Node: id={pid} '{pname}' ===")
        variants = children_of.get(pid, [])
        if not variants:
            print("  (không có con nào -- đây là leaf)")
            continue
        print(f"  Có {len(variants)} biến thể (variant) con:")
        for vid, vweight in variants:
            vname = id_to_name[vid]
            print(f"    -> id={vid} '{vname}'   trong_so = {vweight}")
            leaves = children_of.get(vid, [])
            for lid, lweight in sorted(leaves, key=lambda x: -x[1]):
                lname = id_to_name[lid]
                print(f"         -> id={lid} '{lname}'   P(Anh|Việt) = {lweight}")

# ==== Nhập lemma (hoặc ID số) ở đây ====
show_children_of_lemma("leave*")   # đổi thành lemma/ID bạn muốn tra cứu


=== Node: id=22916 'leave* [lemma]' ===
  Có 4 biến thể (variant) con:
    -> id=12575 'left'   trong_so = 0.751232
         -> id=3932 'tay_trái'   P(Anh|Việt) = 0.391496
         -> id=3333 'cánh_tả'   P(Anh|Việt) = 0.312085
         -> id=3337 'trái'   P(Anh|Việt) = 0.204048
         -> id=905 'rời'   P(Anh|Việt) = 0.202653
         -> id=6632 'bỏ_rơi'   P(Anh|Việt) = 0.151096
         -> id=11308 'dấu_vết'   P(Anh|Việt) = 0.095653
         -> id=4637 'bỏ'   P(Anh|Việt) = 0.069153
    -> id=14548 'leaving'   trong_so = 1.0
         -> id=905 'rời'   P(Anh|Việt) = 0.096426
    -> id=16268 'leave'   trong_so = 1.0
         -> id=10511 'nghỉ_phép'   P(Anh|Việt) = 0.352054
         -> id=905 'rời'   P(Anh|Việt) = 0.175538
         -> id=4959 'nghỉ_việc'   P(Anh|Việt) = 0.116379
         -> id=6095 'yên'   P(Anh|Việt) = 0.113625
         -> id=6632 'bỏ_rơi'   P(Anh|Việt) = 0.106499
         -> id=9755 'ngâm'   P(Anh|Việt) = 0.058717
    -> id=18097 'leaves'   trong_so = 0.453125
       

In [13]:
def print_head(file_path, n):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            for i, line in enumerate(f):
                if i >= n:
                    break
                print(line, end='')
    except FileNotFoundError:
        print("Không tìm thấy file!")

# Nhập tên file và số dòng n từ bàn phím
path = input("Nhập đường dẫn file txt: ")
n_lines = int(input("Nhập số dòng n cần xem: "))

print_head(path, n_lines)


Nhập đường dẫn file txt: name.txt
Nhập số dòng n cần xem: 200
@ITEM=1=nhớ
@ITEM=2=xung_đột
@ITEM=3=thời_tiết
@ITEM=4=nhanh
@ITEM=5=đột_nhiên
@ITEM=6=chân
@ITEM=7=lốp
@ITEM=8=kim_loại
@ITEM=9=tai
@ITEM=10=xe_tải
@ITEM=11=giây
@ITEM=12=tracked
@ITEM=13=wear
@ITEM=14=turned
@ITEM=15=sau
@ITEM=16=một_vài
@ITEM=17=năm
@ITEM=18=chỉ_huy
@ITEM=19=đông
@ITEM=20=trách_nhiệm
@ITEM=21=khoản
@ITEM=22=nội_bộ
@ITEM=23=kiểm_toán
@ITEM=24=kiến_nghị
@ITEM=25=đề_xuất
@ITEM=26=c
@ITEM=27=for
@ITEM=28=point_g
@ITEM=29=tuyệt_vời
@ITEM=30=thu_thập
@ITEM=31=đầy
@ITEM=32=danh_mục
@ITEM=33=cấm
@ITEM=34=thực_vật
@ITEM=35=bảo_vệ
@ITEM=36=việt_nam
@ITEM=37=in
@ITEM=38=chờ
@ITEM=39=chằm_chằm
@ITEM=40=hai
@ITEM=41=ngôi_sao
@ITEM=42=đầu
@ITEM=43=từ_từ
@ITEM=44=thực_sự
@ITEM=45=thích
@ITEM=46=con_người
@ITEM=47=cơ_thể
@ITEM=48=tệ
@ITEM=49=obtained
@ITEM=50=satellite
@ITEM=51=including
@ITEM=52=tham_gia
@ITEM=53=quy_phạm_pháp_luật
@ITEM=54=liên_quan
@ITEM=55=hoạt_động
@ITEM=56=pháp_luật
@ITEM=57=accordance
@ITEM=58=rul